# The Story of Maple — Document Scraper

This notebook scrapes all chapters from [The Story of Maple](https://thestoryofmaple.squarespace.com/main-story-table-of-contents) and saves them as a formatted Word document to your Google Drive.

### Instructions
1. Click **Runtime → Run all** in the menu above (or press `Ctrl+F9`)
2. When prompted, click the link to authorize Google Drive access
3. Wait for it to finish — it will show progress as it goes
4. Find the file at **My Drive → story_of_maple.docx**

The whole process takes around 10–15 minutes.

In [ ]:
# Step 1: Mount Google Drive (so we can save the output there)
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted!')

In [ ]:
# Step 2: Install dependencies
print('Installing dependencies (this takes a minute)...')
!pip install playwright python-docx beautifulsoup4 -q
!playwright install chromium -q
!playwright install-deps chromium -q
print('Done!')

In [ ]:
# Step 3: Run the scraper
import time
import re
from playwright.sync_api import sync_playwright
from bs4 import BeautifulSoup, NavigableString, Tag
from docx import Document
from docx.shared import Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH

BASE_URL = "https://thestoryofmaple.squarespace.com"
TOC_URL = f"{BASE_URL}/main-story-table-of-contents"
OUTPUT_FILE = "/content/drive/My Drive/story_of_maple.docx"

REQUEST_DELAY = 2.0

SKIP_SLUGS = {
    "main-story-table-of-contents",
    "main-story-abridged-table-of-contents",
    "gms-verse-table-of-contents",
    "lore-changes",
    "grandis-plot-points",
    "storyline-order-and-playthroughs",
    "about",
    "contact-me",
    "lore-pre-destiny",
}


def parse_color(style):
    match = re.search(r'color:\s*#([0-9a-fA-F]{6})', style)
    if match:
        hex_color = match.group(1)
        r = int(hex_color[0:2], 16)
        g = int(hex_color[2:4], 16)
        b = int(hex_color[4:6], 16)
        return RGBColor(r, g, b)
    return None


def add_formatted_runs(paragraph, element, inherited_color=None, inherited_bold=False):
    for child in element.children:
        if isinstance(child, NavigableString):
            text = str(child)
            if not text:
                continue
            run = paragraph.add_run(text)
            if inherited_color:
                run.font.color.rgb = inherited_color
            if inherited_bold:
                run.bold = True
        elif isinstance(child, Tag):
            color = inherited_color
            bold = inherited_bold
            style = child.get("style", "")
            if style:
                parsed = parse_color(style)
                if parsed:
                    color = parsed
            if child.name in ("strong", "b"):
                bold = True
            add_formatted_runs(paragraph, child, inherited_color=color, inherited_bold=bold)


def parse_toc(page):
    page.goto(TOC_URL, wait_until="networkidle")
    entries = []
    elements = page.query_selector_all("strong, ol li a, ul li a")
    for el in elements:
        tag = el.evaluate("el => el.tagName.toLowerCase()")
        text = el.inner_text().strip()
        if not text:
            continue
        if tag == "strong" and text.startswith("PART"):
            entries.append({"type": "part", "title": text})
        elif tag == "a":
            href = el.get_attribute("href") or ""
            if href.startswith("/"):
                href = BASE_URL + href
            if BASE_URL in href:
                slug = href.rstrip("/").split("/")[-1]
                if slug not in SKIP_SLUGS:
                    entries.append({"type": "chapter", "title": text, "url": href})
    seen_urls = set()
    deduped = []
    for entry in entries:
        if entry["type"] == "part":
            deduped.append(entry)
        elif entry["url"] not in seen_urls:
            seen_urls.add(entry["url"])
            deduped.append(entry)
    return deduped


def extract_chapter_content(page, url):
    page.goto(url, wait_until="networkidle")
    blocks = []
    content_handles = page.query_selector_all(".sqs-html-content")
    if not content_handles:
        return blocks
    for handle in content_handles:
        html = handle.inner_html()
        soup = BeautifulSoup(html, "html.parser")
        for el in soup.find_all(["h1", "h2", "h3", "h4", "p", "blockquote"]):
            text = el.get_text(strip=True)
            if not text:
                continue
            if el.name in ("h1", "h2", "h3", "h4"):
                blocks.append({"type": "heading", "level": int(el.name[1]), "text": text})
            else:
                blocks.append({"type": "paragraph", "element": el})
    return blocks


def build_document(page, toc_entries):
    doc = Document()
    title_para = doc.add_heading("The Story of Maple", level=0)
    title_para.alignment = WD_ALIGN_PARAGRAPH.CENTER
    doc.add_paragraph("")

    total_chapters = sum(1 for e in toc_entries if e["type"] == "chapter")
    chapter_count = 0

    for entry in toc_entries:
        if entry["type"] == "part":
            doc.add_page_break()
            doc.add_heading(entry["title"], level=1)
            print(f"\n{'='*60}\n  {entry['title']}\n{'='*60}")
        elif entry["type"] == "chapter":
            chapter_count += 1
            print(f"  [{chapter_count}/{total_chapters}] {entry['title']} ...", end=" ", flush=True)
            doc.add_heading(entry["title"], level=2)
            try:
                blocks = extract_chapter_content(page, entry["url"])
            except Exception as e:
                print(f"ERROR: {e}")
                doc.add_paragraph("[Content could not be fetched]")
                continue
            if not blocks:
                doc.add_paragraph("[No content extracted — check this page manually]")
                print("WARNING: no content found")
            else:
                for block in blocks:
                    if block["type"] == "heading":
                        doc.add_heading(block["text"], level=min(block["level"] + 1, 4))
                    else:
                        para = doc.add_paragraph()
                        add_formatted_runs(para, block["element"])
                print(f"OK ({len(blocks)} blocks)")
            time.sleep(REQUEST_DELAY)

    return doc


# --- Run ---
with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    page = browser.new_page()

    print("Fetching table of contents...")
    toc_entries = parse_toc(page)

    chapter_count = sum(1 for e in toc_entries if e["type"] == "chapter")
    part_count = sum(1 for e in toc_entries if e["type"] == "part")
    print(f"Found {part_count} parts and {chapter_count} chapters.\n")

    doc = build_document(page, toc_entries)
    browser.close()

doc.save(OUTPUT_FILE)
print(f"\n✅ Done! File saved to your Google Drive as 'story_of_maple.docx'")